In [2]:
import pandas as pd

# Load JSON directly into a table (DataFrame)
df = pd.read_json("/home/tichar/Documents/ee798/project/results/cross_encoders.json")
df
cols = [c for c in df.columns if "descriptions" in c]
cols


['BAAI/bge-reranker-base (descriptions)',
 'MoritzLaurer/deberta-v3-large-zeroshot-v2.0 (descriptions)',
 'facebook/bart-large-mnli (descriptions)',
 'cross-encoder/nli-deberta-v3-base (descriptions)',
 'MoritzLaurer/bge-m3-zeroshot-v2.0 (descriptions)',
 'MoritzLaurer/deberta-v3-base-zeroshot-v2.0 (descriptions)']

In [3]:
cols = [c for c in df.columns if "descriptions" not in c]
cols

['BAAI/bge-reranker-base',
 'MoritzLaurer/deberta-v3-large-zeroshot-v2.0',
 'facebook/bart-large-mnli',
 'cross-encoder/nli-deberta-v3-base',
 'MoritzLaurer/bge-m3-zeroshot-v2.0',
 'MoritzLaurer/deberta-v3-base-zeroshot-v2.0',
 'MoritzLaurer/deberta-v3-large-zeroshot-v2.0 (labels)',
 'MoritzLaurer/deberta-v3-base-zeroshot-v2.0 (labels)',
 'facebook/bart-large-mnli (labels)',
 'cross-encoder/nli-deberta-v3-base (labels)',
 'MoritzLaurer/bge-m3-zeroshot-v2.0 (labels)']

In [4]:
accuracy = df[cols].apply(lambda col: col.map( lambda cell: cell.get('accuracy') if isinstance(cell, dict) else None ))
accuracy

,BAAI/bge-reranker-base,MoritzLaurer/deberta-v3-large-zeroshot-v2.0,facebook/bart-large-mnli,cross-encoder/nli-deberta-v3-base,MoritzLaurer/bge-m3-zeroshot-v2.0,MoritzLaurer/deberta-v3-base-zeroshot-v2.0,MoritzLaurer/deberta-v3-large-zeroshot-v2.0 (labels),MoritzLaurer/deberta-v3-base-zeroshot-v2.0 (labels),facebook/bart-large-mnli (labels),cross-encoder/nli-deberta-v3-base (labels),MoritzLaurer/bge-m3-zeroshot-v2.0 (labels)
emotion,0.384500,0.700000,0.493000,0.358000,0.697500,0.748000,NaN,NaN,NaN,NaN,NaN
banking77,0.555592,0.721717,0.508127,0.330949,0.653121,0.711964,0.677828,0.653446,0.467165,0.403771,0.663849
commonsense_qa,0.352989,0.536446,0.406224,0.433251,0.339885,0.438165,NaN,NaN,NaN,NaN,NaN
ag_news,0.644211,0.893289,0.674211,0.669342,0.867632,0.877500,NaN,NaN,NaN,NaN,NaN


In [11]:
import pandas as pd

DIR = "/home/tichar/Documents/ee798/project/results/"
FILES = {
    "Bi-encoders":   DIR + "sentence_encoders.json",
    "Cross-encoders": DIR + "cross_encoders.json",
    "Fine-tune-Cross encoders": DIR + "finetuned_cross_encoders.json",   # add when you have it
    "Fine-tune-Bi encoders": DIR + "finetuned_sentence_encoders.json"        # add when you have it
}

exclude = ["descriptions", "checkpoints","labels"]   # keep only plain base runs
DS_NAMES = {"ag_news": "AG News", "banking77": "Banking77",
            "emotion": "Emotion", "commonsense_qa": "CSQA"}  # adjust to your keys

def load_base(path, metric="accuracy"):
    df = pd.read_json(path)
    print(path.split("/")[-1], "->", df.columns.tolist())   # sanity check
    cols = [c for c in df.columns if not any(e in c for e in exclude)]
    t = df[cols].apply(lambda col: col.map(
        lambda x: x.get(metric) if isinstance(x, dict) else None))
    t = (t.T * 100).astype(float).round(1)
    t.index = [m.split("/")[-1] for m in t.index]
    return t.rename(columns=DS_NAMES)

groups = {g: load_base(p) for g, p in FILES.items()}

sentence_encoders.json -> ['BAAI/bge-m3 (descriptions)', 'BAAI/bge-small-en-v1.5 (descriptions)', 'BAAI/bge-small-en-v1.5', 'BAAI/bge-m3', 'sentence-transformers/all-MiniLM-L6-v2', 'intfloat/e5-small-v2', 'intfloat/e5-small-v2(descriptions)', 'sentence-transformers/all-MiniLM-L6-v2(descriptions)']
cross_encoders.json -> ['BAAI/bge-reranker-base', 'BAAI/bge-reranker-base (descriptions)', 'MoritzLaurer/deberta-v3-large-zeroshot-v2.0', 'MoritzLaurer/deberta-v3-large-zeroshot-v2.0 (descriptions)', 'facebook/bart-large-mnli', 'facebook/bart-large-mnli (descriptions)', 'cross-encoder/nli-deberta-v3-base', 'cross-encoder/nli-deberta-v3-base (descriptions)', 'MoritzLaurer/bge-m3-zeroshot-v2.0', 'MoritzLaurer/bge-m3-zeroshot-v2.0 (descriptions)', 'MoritzLaurer/deberta-v3-base-zeroshot-v2.0', 'MoritzLaurer/deberta-v3-base-zeroshot-v2.0 (descriptions)', 'MoritzLaurer/deberta-v3-large-zeroshot-v2.0 (labels)', 'MoritzLaurer/deberta-v3-base-zeroshot-v2.0 (labels)', 'facebook/bart-large-mnli (labels)

In [12]:
DS_NAMES = {"ag_news": "AG News", "emotion": "Emotion",
            "banking77": "Banking77", "commonsense_qa": "CSQA"}   # fixed key

ASSOC  = ["AG News", "Emotion", "Banking77"]
CHANCE = {"AG News": 25.0, "Emotion": 100/6, "Banking77": 100/77, "CSQA": 20.0}

def fancy_latex(groups, caption, label):
    allt = pd.concat(groups.values())[ASSOC + ["CSQA"]].copy()
    allt["Avg"] = allt[ASSOC].mean(axis=1)
    cols = ASSOC + ["Avg", "CSQA"]
    allt = allt[cols]
    best = allt.max()
    second = allt.apply(lambda c: c.dropna().nlargest(2).iloc[-1])

    def cell(v, c):
        if pd.isna(v):
            return "--"
        s = f"{v:.1f}"
        if v == best[c]:
            s = rf"\textbf{{{s}}}"
        elif v == second[c]:
            s = rf"\underline{{{s}}}"
        ch = CHANCE.get(c, sum(CHANCE[a] for a in ASSOC) / 3)
        norm = min(max((v - ch) / (100 - ch), 0), 1)       # chance-normalized
        return rf"\cellcolor{{teal!{int(55 * norm)}}}{s}"

    body = []
    for gi, (g, t) in enumerate(groups.items()):
        if gi > 0:
            body.append(r"\midrule")
        body.append(rf"\multicolumn{{{len(cols)+1}}}{{l}}{{\textit{{{g}}}}} \\")
        for model in t.index:
            row = allt.loc[model]
            name = r"\texttt{" + model.replace("_", r"\_") + "}"
            body.append(rf"\quad {name} & " +
                        " & ".join(cell(row[c], c) for c in cols) + r" \\")

    body.append(r"\midrule")
    chance_cells = [f"{CHANCE[a]:.1f}" for a in ASSOC] + \
                   [f"{sum(CHANCE[a] for a in ASSOC)/3:.1f}", f"{CHANCE['CSQA']:.1f}"]
    body.append(r"\textit{Chance} & " + " & ".join(chance_cells) + r" \\")

    return (
        "\\begin{table*}[t]\n\\centering\\small\n"
        f"\\caption{{{caption}}}\n\\label{{{label}}}\n"
        "\\begin{tabular}{lcccc|c}\n\\toprule\n"
        "& \\multicolumn{4}{c|}{\\textbf{Association tasks}} & \\textbf{Reasoning probe} \\\\\n"
        "\\cmidrule(lr){2-5}\\cmidrule(l){6-6}\n"
        "Model & " + " & ".join(cols) + " \\\\\n\\midrule\n"
        + "\n".join(body) + "\n\\bottomrule\n\\end{tabular}\n\\end{table*}"
    )

print(fancy_latex(groups,
  "Zero-shot accuracy (\\%) with label names only. Shading shows chance-normalized "
  "accuracy; best in bold, second best underlined.", "tab:base"))

\begin{table*}[t]
\centering\small
\caption{Zero-shot accuracy (\%) with label names only. Shading shows chance-normalized accuracy; best in bold, second best underlined.}
\label{tab:base}
\begin{tabular}{lcccc|c}
\toprule
& \multicolumn{4}{c|}{\textbf{Association tasks}} & \textbf{Reasoning probe} \\
\cmidrule(lr){2-5}\cmidrule(l){6-6}
Model & AG News & Emotion & Banking77 & Avg & CSQA \\
\midrule
\multicolumn{6}{l}{\textit{Bi-encoders}} \\
\quad \texttt{bge-small-en-v1.5} & \cellcolor{teal!32}69.3 & \cellcolor{teal!21}49.6 & \cellcolor{teal!37}67.9 & \cellcolor{teal!30}62.3 & \cellcolor{teal!14}40.4 \\
\quad \texttt{bge-m3} & \cellcolor{teal!35}73.1 & \cellcolor{teal!16}41.7 & \cellcolor{teal!35}64.7 & \cellcolor{teal!29}59.8 & \cellcolor{teal!15}42.4 \\
\quad \texttt{all-MiniLM-L6-v2} & \cellcolor{teal!28}64.2 & \cellcolor{teal!17}43.7 & \cellcolor{teal!33}62.3 & \cellcolor{teal!27}56.7 & \cellcolor{teal!13}40.3 \\
\quad \texttt{e5-small-v2} & \cellcolor{teal!31}68.5 & \cellcolor{te

In [13]:
import re

ASSOC = ["AG News", "Emotion", "Banking77"]          # CSQA excluded (choices differ per question)

def load_desc(path, metric="accuracy"):
    df = pd.read_json(path)
    cols = [c for c in df.columns if "descriptions" in c and "checkpoints" not in c]
    t = df[cols].apply(lambda col: col.map(
        lambda x: x.get(metric) if isinstance(x, dict) else None))
    t = (t.T * 100).astype(float).round(1)
    # "BAAI/bge-m3 (descriptions)" -> "bge-m3"
    t.index = [re.sub(r"\s*\(?descriptions?\)?\s*", "", m).split("/")[-1].strip() for m in t.index]
    return t.rename(columns=DS_NAMES)

desc_groups = {g: load_desc(p) for g, p in FILES.items()}

def to_assoc(t):
    t = t.reindex(columns=ASSOC).copy()
    t["Avg"] = t[ASSOC].mean(axis=1)
    return t

# Quick look at the deltas (descriptions minus base), in accuracy points
B = to_assoc(pd.concat(groups.values()))
D = to_assoc(pd.concat(desc_groups.values()))
missing = set(D.index) - set(B.index)
assert not missing, f"names don't match the base table: {missing}"
print((D - B.reindex(D.index)).round(1))

                                AG News  Emotion  Banking77  Avg
bge-m3                              7.7      2.5        4.5  4.9
bge-small-en-v1.5                  11.5     -2.9        2.5  3.7
e5-small-v2                        11.4     -0.2        6.1  5.8
all-MiniLM-L6-v2                   13.3      1.4        4.3  6.3
bge-reranker-base                   7.1      8.3        5.1  6.8
deberta-v3-large-zeroshot-v2.0     -0.2     -1.3       -4.8 -2.1
bart-large-mnli                    11.5      5.9       -6.8  3.5
nli-deberta-v3-base                -6.3      5.4        6.7  1.9
bge-m3-zeroshot-v2.0                2.0     -1.4       -5.7 -1.7
deberta-v3-base-zeroshot-v2.0       1.8     -6.2       -7.7 -4.0


In [14]:
def desc_latex(desc_groups, base_groups, caption, label):
    cols = ASSOC + ["Avg"]
    D = to_assoc(pd.concat(desc_groups.values()))
    B = to_assoc(pd.concat(base_groups.values()))
    best = D.max()

    def cell(v, d, c):
        if pd.isna(v):
            return "--"
        s = f"{v:.1f}"
        if v == best[c]:
            s = rf"\textbf{{{s}}}"
        if pd.isna(d):
            return s
        shade = min(int(4 * abs(d)), 50)                    # 12.5 points or more saturates
        colour = "green" if d >= 0 else "red"
        return rf"\cellcolor{{{colour}!{shade}}}{s}\,{{\scriptsize({d:+.1f})}}"

    body = []
    for gi, (g, t) in enumerate(desc_groups.items()):
        if gi > 0:
            body.append(r"\midrule")
        body.append(rf"\multicolumn{{{len(cols)+1}}}{{l}}{{\textit{{{g}}}}} \\")
        for model in t.index:
            name = r"\texttt{" + model.replace("_", r"\_") + "}"
            cells = []
            for c in cols:
                base_v = B.loc[model, c] if model in B.index else float("nan")
                cells.append(cell(D.loc[model, c], D.loc[model, c] - base_v, c))
            body.append(rf"\quad {name} & " + " & ".join(cells) + r" \\")

    return (
        "\\begin{table*}[t]\n\\centering\\small\n"
        f"\\caption{{{caption}}}\n\\label{{{label}}}\n"
        "\\begin{tabular}{lccc|c}\n\\toprule\n"
        "Model & " + " & ".join(cols) + " \\\\\n\\midrule\n"
        + "\n".join(body) + "\n\\bottomrule\n\\end{tabular}\n\\end{table*}"
    )

print(desc_latex(desc_groups, groups,
  "Zero-shot accuracy (\\%) with class descriptions. Bracketed values are the change "
  "from label names only (Table~\\ref{tab:base}); green marks gains, red marks losses. "
  "Best in bold. Avg is the raw mean over the three tasks.",
  "tab:desc"))

\begin{table*}[t]
\centering\small
\caption{Zero-shot accuracy (\%) with class descriptions. Bracketed values are the change from label names only (Table~\ref{tab:base}); green marks gains, red marks losses. Best in bold. Avg is the raw mean over the three tasks.}
\label{tab:desc}
\begin{tabular}{lccc|c}
\toprule
Model & AG News & Emotion & Banking77 & Avg \\
\midrule
\multicolumn{5}{l}{\textit{Bi-encoders}} \\
\quad \texttt{bge-m3} & \cellcolor{green!30}80.8\,{\scriptsize(+7.7)} & \cellcolor{green!10}44.2\,{\scriptsize(+2.5)} & \cellcolor{green!18}69.2\,{\scriptsize(+4.5)} & \cellcolor{green!19}64.7\,{\scriptsize(+4.9)} \\
\quad \texttt{bge-small-en-v1.5} & \cellcolor{green!46}80.8\,{\scriptsize(+11.5)} & \cellcolor{red!11}46.7\,{\scriptsize(-2.9)} & \cellcolor{green!10}\textbf{70.4}\,{\scriptsize(+2.5)} & \cellcolor{green!14}66.0\,{\scriptsize(+3.7)} \\
\quad \texttt{e5-small-v2} & \cellcolor{green!45}79.9\,{\scriptsize(+11.4)} & \cellcolor{red!0}46.2\,{\scriptsize(-0.2)} & \cellcolo

In [15]:
import re

def load_ft(path, metric="accuracy"):
    df = pd.read_json(path)
    rows = []
    for col in [c for c in df.columns if "checkpoint" in c]:
        for ds, cell in df[col].items():
            if isinstance(cell, dict):
                short = col.split("/")[-1]
                model = re.sub(rf"[-_]{re.escape(ds)}$", "", short)      # "bge-small-ag_news" -> "bge-small"
                rows.append({"ckpt": col, "model": model,
                             "dataset": DS_NAMES.get(ds, ds), "acc": cell[metric] * 100})
    return pd.DataFrame(rows)

ft = pd.concat([load_ft(p).assign(group=g) for g, p in FILES.items()], ignore_index=True)

# match checkpoint model names to base-table names by prefix
base_all = pd.concat(groups.values())
def match_base(m):
    idx = list(base_all.index)
    exact = [b for b in idx if b.lower() == m.lower()]
    if exact:
        return exact[0]
    hits = [b for b in idx if b.lower().startswith(m.lower())]
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        print(f"ambiguous: {m} -> {hits}")
    return None
ft["base_model"] = ft["model"].map(match_base)
print("unmatched:", ft[ft.base_model.isna()].ckpt.unique())             # fix these by hand if any
ft = ft.dropna(subset=["base_model"])

FT_COLS = ["AG News", "Emotion", "Banking77", "CSQA"]
acc = ft.pivot_table(index="base_model", columns="dataset", values="acc").reindex(columns=FT_COLS)
base = base_all.reindex(index=acc.index, columns=FT_COLS)
delta = acc - base
print(delta.round(1))

unmatched: <ArrowStringArray>
[]
Length: 0, dtype: str
dataset                        AG News  Emotion  Banking77  CSQA
base_model                                                      
all-MiniLM-L6-v2                  24.3     17.1        6.1  -0.3
bge-small-en-v1.5                 21.7     15.5        6.4   0.8
deberta-v3-base-zeroshot-v2.0      4.0      8.8        5.8  12.6
e5-small-v2                       22.9     21.2        9.4   6.0
nli-deberta-v3-base               24.2     32.6       32.6   9.9


In [16]:
import json
import re
from pathlib import Path

FT_FILES = {
    "Fine-tuned cross-encoders": FILES["Fine-tune-Cross encoders"],
    "Fine-tuned bi-encoders": FILES["Fine-tune-Bi encoders"],
}

DS_NAMES = {
    "ag_news": "AG News",
    "emotion": "Emotion",
    "banking77": "Banking77",
    "commonsense_qa": "CSQA",
}

DATASET_SUFFIXES = {
    "ag_news": ("ag_news", "ag-news", "agnews"),
    "emotion": ("emotion",),
    "banking77": ("banking77",),
    "commonsense_qa": ("commonsense_qa", "commonsense-qa", "commonsenseqa"),
}

def strip_dataset_suffix(checkpoint_name, dataset):
    for suffix in sorted(DATASET_SUFFIXES.get(dataset, (dataset,)), key=len, reverse=True):
        model = re.sub(rf"[-_]{re.escape(suffix)}$", "", checkpoint_name, flags=re.IGNORECASE)
        if model != checkpoint_name:
            return model
    return checkpoint_name

def load_ft(path, group, metric="accuracy"):
    with open(path, encoding="utf-8") as f:
        results = json.load(f)

    rows = []
    for checkpoint, dataset_results in results.items():
        if not isinstance(dataset_results, dict):
            continue

        checkpoint_name = Path(checkpoint).name

        for dataset, metrics in dataset_results.items():
            if not isinstance(metrics, dict) or metric not in metrics:
                continue

            model = strip_dataset_suffix(checkpoint_name, dataset)
            rows.append({
                "group": group,
                "model": model,
                "checkpoint": checkpoint,
                "dataset": DS_NAMES.get(dataset, dataset),
                "accuracy": metrics[metric] * 100,
            })

    return pd.DataFrame(rows)

ft_long = pd.concat(
    [load_ft(path, group) for group, path in FT_FILES.items()],
    ignore_index=True,
)

# One row per checkpoint naming variant; accuracy is shown as a percentage.
ft_accuracy = (
    ft_long.pivot_table(
        index=["group", "model"],
        columns="dataset",
        values="accuracy",
        aggfunc="first",
    )
    .reindex(columns=["AG News", "Emotion", "Banking77", "CSQA"])
    .round(1)
)

ft_accuracy

dataset                                                  AG News  Emotion  \
group                     model                                             
Fine-tuned bi-encoders    all-MiniLM-L6-v2                  88.5     60.8   
                          bge-small                         91.0     65.1   
                          e5-small-v2                       91.4     67.6   
Fine-tuned cross-encoders deberta-v3-base-zeroshot-v2.0     91.8     83.6   
                          nli-deberta-v3-base               91.1     68.4   

dataset                                                  Banking77  CSQA  
group                     model                                           
Fine-tuned bi-encoders    all-MiniLM-L6-v2                    68.4  40.0  
                          bge-small                           74.3  41.2  
                          e5-small-v2                         68.3  47.1  
Fine-tuned cross-encoders deberta-v3-base-zeroshot-v2.0       77.0  56.4  
                          nli-deberta-v3-base                 65.7  53.2

In [17]:
import pandas as pd

FT_COLS = ["AG News", "Emotion", "Banking77", "CSQA"]
ASSOC = ["AG News", "Emotion", "Banking77"]
FT_TABLE_COLS = ASSOC + ["Avg", "CSQA"]
CHANCE = {
    "AG News": 25.0,
    "Emotion": 100 / 6,
    "Banking77": 100 / 77,
    "CSQA": 20.0,
    "Avg": sum([25.0, 100 / 6, 100 / 77]) / 3,
}

# Compare against the corresponding zero-shot bi- or cross-encoder.
base_all = pd.concat([groups["Bi-encoders"], groups["Cross-encoders"]])
base_all = base_all.reindex(columns=FT_COLS)

def match_base_model(model):
    names = list(base_all.index)
    exact = [name for name in names if name.lower() == model.lower()]
    if exact:
        return exact[0]
    matches = [name for name in names if name.lower().startswith(model.lower())]
    if len(matches) == 1:
        return matches[0]
    print(f"Could not uniquely match fine-tuned model {model!r} to a base model: {matches}")
    return None

ft_table = ft_accuracy.copy()
ft_table["base_model"] = [match_base_model(model) for _, model in ft_table.index]
ft_table["Avg"] = ft_table[ASSOC].mean(axis=1)
base_for_ft = base_all.reindex(ft_table["base_model"].to_list())
base_for_ft.index = ft_table.index
base_for_ft["Avg"] = base_for_ft[ASSOC].mean(axis=1)
ft_delta = ft_table[FT_TABLE_COLS] - base_for_ft[FT_TABLE_COLS]


def fine_tuned_latex(table, delta, caption, label):
    best = table.max(axis=0)
    body = []
    groups_in_table = table.index.get_level_values("group").unique()

    def format_cell(value, difference, column):
        if pd.isna(value):
            return "--"

        text = f"{value:.1f}"
        if value == best[column]:
            text = rf"\textbf{{{text}}}"

        if pd.notna(difference):
            text += rf"\,{{\scriptsize({difference:+.1f})}}"

        chance = CHANCE[column]
        normalized = min(max((value - chance) / (100 - chance), 0), 1)
        return rf"\cellcolor{{teal!{int(55 * normalized)}}}{text}"

    for group_index, group in enumerate(groups_in_table):
        if group_index:
            body.append(r"\midrule")

        body.append(rf"\multicolumn{{{len(FT_TABLE_COLS) + 1}}}{{l}}{{\textit{{{group}}}}} \\")
        group_table = table.xs(group, level="group")
        group_delta = delta.xs(group, level="group")
        for model, row in group_table.iterrows():
            model_name = str(model).replace("_", r"\_")
            cells = " & ".join(
                format_cell(row[column], group_delta.loc[model, column], column)
                for column in FT_TABLE_COLS
            )
            body.append(rf"\quad \texttt{{{model_name}}} & {cells} \\")

    chance_cells = " & ".join(f"{CHANCE[column]:.1f}" for column in FT_TABLE_COLS)
    body.extend([
        r"\midrule",
        rf"\textit{{Chance}} & {chance_cells} \\",
    ])

    return (
        "\\begin{table*}[t]\n\\centering\\small\n"
        f"\\caption{{{caption}}}\n\\label{{{label}}}\n"
        "\\begin{tabular}{lcccc|c}\n\\toprule\n"
        "Model & " + " & ".join(FT_TABLE_COLS) + " \\\\\n\\midrule\n"
        + "\n".join(body)
        + "\n\\bottomrule\n\\end{tabular}\n\\end{table*}"
    )

print(fine_tuned_latex(
    ft_table,
    ft_delta,
    "Fine-tuned model accuracy (\\%). Parentheses show the accuracy-point change from the corresponding zero-shot model; Avg is over AG News, Emotion, and Banking77.",
    "tab:finetuned",
))

\begin{table*}[t]
\centering\small
\caption{Fine-tuned model accuracy (\%). Parentheses show the accuracy-point change from the corresponding zero-shot model; Avg is over AG News, Emotion, and Banking77.}
\label{tab:finetuned}
\begin{tabular}{lcccc|c}
\toprule
Model & AG News & Emotion & Banking77 & Avg & CSQA \\
\midrule
\multicolumn{6}{l}{\textit{Fine-tuned bi-encoders}} \\
\quad \texttt{all-MiniLM-L6-v2} & \cellcolor{teal!46}88.5\,{\scriptsize(+24.3)} & \cellcolor{teal!29}60.8\,{\scriptsize(+17.1)} & \cellcolor{teal!37}68.4\,{\scriptsize(+6.1)} & \cellcolor{teal!37}72.6\,{\scriptsize(+15.8)} & \cellcolor{teal!13}40.0\,{\scriptsize(-0.3)} \\
\quad \texttt{bge-small} & \cellcolor{teal!48}91.0\,{\scriptsize(+21.7)} & \cellcolor{teal!31}65.1\,{\scriptsize(+15.5)} & \cellcolor{teal!40}74.3\,{\scriptsize(+6.4)} & \cellcolor{teal!40}76.8\,{\scriptsize(+14.5)} & \cellcolor{teal!14}41.2\,{\scriptsize(+0.8)} \\
\quad \texttt{e5-small-v2} & \cellcolor{teal!48}91.4\,{\scriptsize(+22.9)} & \cell

In [7]:
import json
from pathlib import Path

RESULTS = Path("/home/tichar/Documents/ee798/project/results/reasoning_csqa.json")
OUT = Path("/home/tichar/Documents/ee798/project/results/reasoning_table.tex")

DISPLAY = {
    "sentence-transformers/all-MiniLM-L6-v2": "all-MiniLM-L6-v2",
    "intfloat/e5-small-v2": "e5-small-v2",
    "BAAI/bge-small-en-v1.5": "bge-small-en-v1.5",
    "BAAI/bge-m3": "bge-m3",
    "MoritzLaurer/deberta-v3-large-zeroshot-v2.0": "DeBERTa-L-ZS",
    "MoritzLaurer/deberta-v3-base-zeroshot-v2.0": "DeBERTa-B-ZS",
    "facebook/bart-large-mnli": "BART-large-MNLI",
    "cross-encoder/nli-deberta-v3-base": "NLI-DeBERTa-B",
    "MoritzLaurer/bge-m3-zeroshot-v2.0": "bge-m3-ZS",
    "BAAI/bge-reranker-base": "bge-reranker-base",
}
GROUPS = [("Bi-encoders", "sentence_encoder"),
          ("Cross-encoders", "nli"),
          ("Reranker", "reranker")]
CHANCE = 20.0
SHADE_SCALE, SHADE_CAP = 3, 45
NCOLS = 8


def esc(s):
    return s.replace("_", r"\_")


def signed(x, nd=1):
    return f"{x:+.{nd}f}".replace("-", "$-$")


def shade(delta):
    if delta is None or abs(delta) < 0.05:
        return ""
    color = "teal" if delta > 0 else "red"
    return rf"\cellcolor{{{color}!{min(SHADE_CAP, int(round(abs(delta) * SHADE_SCALE)))}}}"


def delta_cell(d, sig=False):
    if d is None:
        return "--"
    return f"{shade(d)}{signed(d)}" + (r"$^{\dagger}$" if sig else "")


def acc_cell(v, best):
    s = f"{v:.1f}"
    return rf"\textbf{{{s}}}" if abs(v - best) < 1e-9 else s


data = json.loads(RESULTS.read_text())
data.pop("metadata", None)

rows = {}
for name, m in data.items():
    if "error" in m:
        print(f"skipping {name}: {m['error']}")
        continue
    c, cmp_ = m["conditions"], m["comparisons"]["nonleak"]
    base, rat, shuf = (c[k]["acc"] * 100 for k in ("base", "rat", "shuf"))
    nb, nr = c["base"]["acc_nonleak"], c["rat"]["acc_nonleak"]
    p = cmp_["rat_vs_base_p"]
    rows[name] = dict(
        type=m["model_type"], base=base, rat=rat, shuf=shuf,
        d_all=rat - base,
        d_f=(nr - nb) * 100 if nb is not None and nr is not None else None,
        sig=(p is not None and p < 0.05),
        mb=c["base"]["norm_margin_mean"], mr=c["rat"]["norm_margin_mean"],
    )

best_base = max(r["base"] for r in rows.values())
best_rat = max(r["rat"] for r in rows.values())

L = []
add = L.append
add(r"\begin{table}[t]")
add(r"\centering\footnotesize")
add(r"\setlength{\tabcolsep}{3pt}")
add(r"\caption{Rationale-assisted CSQA accuracy (\%). Base: question only; Rat: question plus an "
    r"LLM-generated rationale; Shuf: question plus the rationale of a different question (control). "
    r"$\Delta$: accuracy change of Rat over Base in points, on all items and on the filtered subset "
    r"of items whose rationale does not mention the correct option (teal: gain, red: loss). "
    r"Norm.\ margin: mean of $(s_{\mathrm{correct}}-\max s_{\mathrm{wrong}})/\mathrm{std}(s)$. "
    r"$\dagger$: Rat vs.\ Base significant on the filtered subset (exact McNemar, $p<0.05$). "
    r"Best accuracy per column in bold.}")
add(r"\label{tab:rationale}")
add(r"\begin{tabularx}{\columnwidth}{@{}>{\raggedright\arraybackslash}X" + "c" * (NCOLS - 1) + r"@{}}")
add(r"\toprule")
add(r"& \multicolumn{3}{c}{Accuracy} & \multicolumn{2}{c}{$\Delta$ Rat$-$Base} "
    r"& \multicolumn{2}{c}{Norm.\ margin} \\")
add(r"\cmidrule(lr){2-4}\cmidrule(lr){5-6}\cmidrule(l){7-8}")
add(r"Model & Base & Rat & Shuf & All & Filtered & Base & Rat \\")
add(r"\midrule")

for title, mtype in GROUPS:
    grp = [(n, r) for n, r in rows.items() if r["type"] == mtype]
    if not grp:
        continue
    add(rf"\multicolumn{{{NCOLS}}}{{l}}{{\textit{{{title}}}}} \\")
    for n, r in grp:
        add(rf"\quad \texttt{{{esc(DISPLAY.get(n, n.split('/')[-1]))}}} & "
            rf"{acc_cell(r['base'], best_base)} & {acc_cell(r['rat'], best_rat)} & {r['shuf']:.1f} & "
            rf"{delta_cell(r['d_all'])} & {delta_cell(r['d_f'], r['sig'])} & "
            rf"{signed(r['mb'], 2)} & {signed(r['mr'], 2)} \\")
    add(r"\midrule")

add(rf"\textit{{Chance}} & {CHANCE:.1f} & {CHANCE:.1f} & {CHANCE:.1f} & -- & -- & -- & -- \\")
add(r"\bottomrule")
add(r"\end{tabularx}")
add(r"\end{table}")

tex = "\n".join(L)
OUT.write_text(tex)
print(tex)
print(f"\nsaved {OUT}")

\begin{table}[t]
\centering\footnotesize
\setlength{\tabcolsep}{3pt}
\caption{Rationale-assisted CSQA accuracy (\%). Base: question only; Rat: question plus an LLM-generated rationale; Shuf: question plus the rationale of a different question (control). $\Delta$: accuracy change of Rat over Base in points, on all items and on the filtered subset of items whose rationale does not mention the correct option (teal: gain, red: loss). Norm.\ margin: mean of $(s_{\mathrm{correct}}-\max s_{\mathrm{wrong}})/\mathrm{std}(s)$. $\dagger$: Rat vs.\ Base significant on the filtered subset (exact McNemar, $p<0.05$). Best accuracy per column in bold.}
\label{tab:rationale}
\begin{tabularx}{\columnwidth}{@{}>{\raggedright\arraybackslash}Xccccccc@{}}
\toprule
& \multicolumn{3}{c}{Accuracy} & \multicolumn{2}{c}{$\Delta$ Rat$-$Base} & \multicolumn{2}{c}{Norm.\ margin} \\
\cmidrule(lr){2-4}\cmidrule(lr){5-6}\cmidrule(l){7-8}
Model & Base & Rat & Shuf & All & Filtered & Base & Rat \\
\midrule
\multicolumn{

In [8]:
import json
import random
from pathlib import Path

from datasets import load_dataset
from src.configs import DATASETS
from src.dataset_adapters import DATASET_ADAPTERS

SEED = 0
OUT = Path("results/dataset_examples.tex")
DATA_CACHE = "/home/tichar/Documents/ee798/project/data"
DISPLAY = {"ag_news": "AG News", "banking77": "Banking77",
           "emotion": "Emotion", "commonsense_qa": "CSQA"}
ORDER = ["ag_news", "banking77", "emotion", "commonsense_qa"]   # edit to match your config keys
MAX_CHARS = 140                                                  # truncate long inputs in the table


def esc(s):
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("$", r"\$"),
                 ("#", r"\#"), ("_", r"\_"), ("{", r"\{"), ("}", r"\}"), ("~", r"\textasciitilde{}"),
                 ("^", r"\textasciicircum{}")]:
        s = s.replace(a, b)
    return s


def clip(s):
    s = " ".join(s.split())
    return s if len(s) <= MAX_CHARS else s[:MAX_CHARS].rsplit(" ", 1)[0] + "..."


rng = random.Random(SEED)
rows = []

for key in ORDER:
    if key not in DATASETS or key not in DATASET_ADAPTERS:
        print(f"skipping {key}: not in DATASETS / DATASET_ADAPTERS (keys: {list(DATASETS)})")
        continue
    config = DATASETS[key]
    dataset = load_dataset(config["hf_name"], cache_dir=DATA_CACHE)
    queries, candidates, labels = DATASET_ADAPTERS[key](dataset, config)

    i = rng.randrange(len(queries))
    opts = candidates[0] if config["shared_candidates"] else candidates[i]
    gold = opts[labels[i]]

    print(f"\n=== {DISPLAY.get(key, key)} | item {i} of {len(queries)} ===")
    print("input :", queries[i])
    print("gold  :", gold)
    print("K     :", len(opts), "| options:", opts if len(opts) <= 10 else opts[:10] + ["..."])

    if config["shared_candidates"]:
        text = esc(clip(queries[i]))
    else:  # per-item options (CSQA): show them with the correct one in bold
        shown = ", ".join(rf"\textbf{{{esc(o)}}}" if j == labels[i] else esc(o)
                          for j, o in enumerate(opts))
        text = f"{esc(clip(queries[i]))} Options: {shown}"
    rows.append((DISPLAY.get(key, key), text, esc(gold)))

L = [r"\begin{table}[htbp]", r"\centering", r"\footnotesize",
     r"\caption{Example inputs and correct labels, one random item per dataset "
     r"(CSQA: correct option in bold).}",
     r"\label{tab:examples}",
     r"\begin{tabularx}{\columnwidth}{@{}l>{\raggedright\arraybackslash}X>{\raggedright\arraybackslash}p{2.1cm}@{}}",
     r"\toprule", r"Dataset & Input & Label \\", r"\midrule"]
for name, text, gold in rows:
    L.append(rf"{name} & ``{text}'' & {gold} \\" if name != "CSQA"
             else rf"{name} & ``{text.split(' Options: ')[0]}'' Options: {text.split(' Options: ')[1]} & {gold} \\")
L += [r"\bottomrule", r"\end{tabularx}", r"\end{table}"]

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text("\n".join(L))
print("\n" + "\n".join(L))
print(f"\nsaved {OUT}")

/home/tichar/Documents/ee798/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== AG News | item 6917 of 7600 ===
input : Allianz to fight US court ruling on WTC attacks MUNICH - German insurance concern Allianz said on Tuesday it would fight a US jury decision in New York which doubled the amount of insurance which the leaseholder of the destroyed World Trade Center towers could collect from nine insurance firms.
gold  : Business
K     : 4 | options: ['World', 'Sports', 'Business', 'Sci/Tech']

=== Banking77 | item 1577 of 3076 ===
input : My ATM transaction was unsuccessful.
gold  : declined cash withdrawal
K     : 77 | options: ['activate my card', 'age limit', 'apple pay or google pay', 'atm support', 'automatic top up', 'balance not updated after bank transfer', 'balance not updated after cheque or cash deposit', 'beneficiary not allowed', 'cancel transfer', 'card about to expire', '...']

=== Emotion | item 1552 of 2000 ===
input : i am finally starting to feel like i have a real life here in san vicente and i am no longer on a strange confusing extended 

In [9]:
import re
import random
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from src.configs import DATASETS
from src.dataset_adapters import DATASET_ADAPTERS

SEED = 0
DATA_CACHE = "/home/tichar/Documents/ee798/project/data"
OUT = Path("results/qual_examples_todo.tex")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMO_KEY, CSQA_KEY = "emotion", "commonsense_qa"       # edit to match your DATASETS keys
EMO_TEMPLATE = "This example is {}."                   # use the template you actually used for Emotion
CSQA_TEMPLATE = "The answer to the question is {}."
MARGIN_MIN, MAX_Q_CHARS, POOL = 0.8, 100, 300          # POOL: max candidates scored per dataset

# ---------- scorers (fp32, same as the main experiments) ----------
e5_tok = AutoTokenizer.from_pretrained("intfloat/e5-small-v2")
e5 = AutoModel.from_pretrained("intfloat/e5-small-v2").to(DEVICE).eval()

def e5_enc(texts):
    b = e5_tok(["query: " + t for t in texts], padding=True, truncation=True,
               max_length=512, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        h = e5(**b).last_hidden_state
    m = b["attention_mask"].unsqueeze(-1).float()
    return F.normalize((h * m).sum(1) / m.sum(1), dim=-1)

def e5_scores(q, opts):
    return (e5_enc([q]) @ e5_enc(opts).T)[0].cpu().numpy()

NLI = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
ce_tok = AutoTokenizer.from_pretrained(NLI)
ce = AutoModelForSequenceClassification.from_pretrained(NLI).to(DEVICE).eval()
ENT = next(int(i) for i, l in ce.config.id2label.items() if str(l).lower() == "entailment")

def ce_scores(q, opts, template):
    hyps = [template.format(o) for o in opts]
    b = ce_tok([q] * len(opts), hyps, padding=True, truncation=True,
               max_length=512, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        return ce(**b).logits[:, ENT].float().cpu().numpy()

def norm_margin(s, g):
    return (s[g] - np.delete(s, g).max()) / (s.std() + 1e-9)

def load(key):
    cfg = DATASETS[key]
    ds = load_dataset(cfg["hf_name"], cache_dir=DATA_CACHE)
    q, c, y = DATASET_ADAPTERS[key](ds, cfg)
    return cfg, q, c, y

def esc(s):
    for a, b in [("&", r"\&"), ("%", r"\%"), ("$", r"\$"), ("#", r"\#"), ("_", r"\_")]:
        s = s.replace(a, b)
    return s

rng = random.Random(SEED)
rows = []

# ================= Emotion: explicit =================
LEX = {
    "sadness": ["sad", "sadness", "unhappy", "depressed", "miserable", "sorrow", "grief", "heartbroken"],
    "joy": ["happy", "joy", "joyful", "delighted", "glad", "cheerful", "pleased"],
    "love": ["love", "loved", "loving", "affection", "adore", "fond"],
    "anger": ["angry", "anger", "furious", "mad", "irritated", "annoyed", "rage"],
    "fear": ["afraid", "fear", "scared", "terrified", "frightened"],
    "surprise": ["surprised", "surprise", "shocked", "amazed", "astonished", "stunned"],
}

def explicit(text, label):
    words = LEX.get(label.lower(), []) + [label.lower()]
    return any(re.search(rf"\b{re.escape(w)}(s|ed)?\b", text.lower()) for w in words)

cfg, Q, C, Y = load(EMO_KEY)
labels = C[0] if cfg["shared_candidates"] else None
cand = [i for i in range(len(Q)) if explicit(Q[i], (labels or C[i])[Y[i]])]
print(f"Emotion: {len(cand)}/{len(Q)} items contain the label word or a synonym")
rng.shuffle(cand)

hits = []
for i in cand[:POOL]:
    opts = labels or C[i]
    g = Y[i]
    s1, s2 = e5_scores(Q[i], opts), ce_scores(Q[i], opts, EMO_TEMPLATE)
    if s1.argmax() == g and s2.argmax() == g:
        hits.append((i, s1, s2))
print(f"  {len(hits)} of {min(POOL, len(cand))} checked items are correct for both scorers")

for k, (i, s1, s2) in enumerate(hits[:5]):
    opts = labels or C[i]
    print(f"\n[Emotion cand {k}] item {i}: {Q[i]!r}\n  label: {opts[Y[i]]}")
    print("  e5  :", {o: round(float(v), 3) for o, v in zip(opts, s1)})
    print("  deb :", {o: round(float(v), 2) for o, v in zip(opts, s2)})
if hits:
    i = hits[0][0]
    opts = labels or C[i]
    rows.append(("Emotion (explicit)", rf"``{esc(Q[i])}''", esc(opts[Y[i]])))

# ================= CSQA: easy association =================
cfg, Q, C, Y = load(CSQA_KEY)
cand = [i for i in range(len(Q)) if len(Q[i]) <= MAX_Q_CHARS]
rng.shuffle(cand)

hits = []
for i in cand[:POOL]:
    g = Y[i]
    s1, s2 = e5_scores(Q[i], C[i]), ce_scores(Q[i], C[i], CSQA_TEMPLATE)
    if (s1.argmax() == g and s2.argmax() == g
            and norm_margin(s1, g) >= MARGIN_MIN and norm_margin(s2, g) >= MARGIN_MIN):
        hits.append((i, s1, s2))
print(f"\nCSQA: {len(hits)} of {min(POOL, len(cand))} short items are easy for both scorers")

for k, (i, s1, s2) in enumerate(hits[:5]):
    print(f"\n[CSQA cand {k}] item {i}: {Q[i]!r}")
    for j, o in enumerate(C[i]):
        print(f"  {'*' if j == Y[i] else ' '} {o:25s} e5={s1[j]:.3f}  deb={s2[j]:+.2f}")
if hits:
    i = hits[0][0]
    shown = ", ".join(rf"\textbf{{{esc(o)}}}" if j == Y[i] else esc(o) for j, o in enumerate(C[i]))
    rows.append(("CSQA (easy)", rf"``{esc(Q[i])}'' Options: {shown}", esc(C[i][Y[i]])))

# ---------- LaTeX rows ----------
tex = "\n".join(rf"{n} & {t} & {l} \\" for n, t, l in rows)
OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(tex)
print("\n--- LaTeX rows (paste into the examples table) ---\n" + tex)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3012.89it/s]
/home/tichar/Documents/ee798/project/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 3651.83it/s]


Emotion: 197/2000 items contain the label word or a synonym
  153 of 197 checked items are correct for both scorers

[Emotion cand 0] item 957: 'i wonder how it feels to be loved by someone you love'
  label: love
  e5  : {'sadness': 0.811, 'joy': 0.795, 'love': 0.865, 'anger': 0.798, 'fear': 0.799, 'surprise': 0.764}
  deb : {'sadness': -2.1, 'joy': -2.77, 'love': 2.11, 'anger': -4.75, 'fear': -4.86, 'surprise': -1.82}

[Emotion cand 1] item 1650: 'i didint feel any love and caring now'
  label: love
  e5  : {'sadness': 0.751, 'joy': 0.766, 'love': 0.822, 'anger': 0.749, 'fear': 0.747, 'surprise': 0.744}
  deb : {'sadness': -2.21, 'joy': -4.29, 'love': 2.62, 'anger': -4.46, 'fear': -4.86, 'surprise': -1.3}

[Emotion cand 2] item 626: 'i remember feeling so frightened that i could feel emotions at that high a level'
  label: fear
  e5  : {'sadness': 0.787, 'joy': 0.739, 'love': 0.744, 'anger': 0.802, 'fear': 0.824, 'surprise': 0.743}
  deb : {'sadness': -4.75, 'joy': -4.95, 'love': -4.